# Fig. 2C, fig. S2H: excitatory vs inhibitory 3'UTRome across human cortical-organoid neurogenesis

Author: Andre Macedo Date: 2026-07-01 Kernel: `apa-python`

Purpose: locate where along the two organoid neurogenic lineages (Knoblich CHOOSE data) the
3'UTR-length programme ("UTRome") diverges between excitatory (EXC) and inhibitory (INH)
neurons, i.e. which cell types and pseudotime stages carry the length differences.

Cross-lineage counterpart of the mouse notebook
`analysis/mouse_svz/13_260701-correlations_over_pseudotime.ipynb`: it reuses
the same pseudobulk W 3'UTR-length index (`W = sum_k rank_k * pi_k` from within-gene PAS
proportions, so W is gene-count-independent; higher W = longer/more-distal 3'UTR), but instead of
transcriptome-vs-UTRome it compares EXC-UTRome vs INH-UTRome. Uninduced (control) cells only, so
the comparison reflects normal development, without the CRISPR perturbation.

Read-outs: (1) cross-lineage cell-type W-correlation map, (2) cross-lineage pseudotime-bin W map with
cell-type strips, (3) global 3'UTR-length trajectory per lineage, (4) divergence-by-stage
(similarity + magnitude), (5) per-gene Delta-W localisation (which genes diverge, and where).
Panels are written to `Fig2/unassigned_panels/`.


## 1. Setup


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import subprocess
import sys
import warnings
from collections import defaultdict
from pathlib import Path

import anndata as ad
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch
from scipy.sparse import issparse
from scipy.stats import rankdata, spearmanr

warnings.filterwarnings("ignore")

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

# ---- paths ----------------------------------------------------------------
KN_BASE = Path(f"{DATA_ROOT}/3utr/organoids/knoblich")
UTR_H5 = KN_BASE / "utrcalling/merged_bw12_above10cts_apa_named_annotated_fix_pt.h5ad"
UNASSIGNED_DIR = S.OUT_ROOT / "Fig2" / "unassigned_panels"
UNASSIGNED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = KN_BASE / "utrcalling" / "lineage_utrome_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- constants ------------------------------------------------------------
CONTROLS_ONLY = True  # uninduced cells (clean developmental comparison)
N_BINS = 12  # equal-frequency pseudotime bins per lineage (matched)
MIN_READS_CT = 25  # min summed UMIs per (gene, cell type)
MIN_READS_BIN = 15  # min summed UMIs per (gene, pseudotime bin)
TOP_N_FEATURES = 1000  # top-variance genes for correlation heatmaps
CORR_CMAP = "magma"
EXC_ORDER = S.ORGANOID_EXC_ORDER  # oRG -> ... -> L23 (progenitor -> neuron)
INH_ORDER = S.ORGANOID_INH_ORDER  # vRG -> ... -> CGE_IN
PAL = S.paper_palette_organoid_celltypes
LIN_COL = {"EXC": S.paper_palette_lineage["EXC"], "INH": S.paper_palette_lineage["INH"]}
print("setup ready; panels ->", UNASSIGNED_DIR)

## 2. Load data and split the two lineages

Uninduced cells with a valid per-lineage diffusion pseudotime. Excitatory cells use
`dpt_pseudotime_excitatory`; inhibitory cells use `dpt_pseudotime_inhibitory`. Astrocytes (in neither
lineage) are dropped. Both modalities share the same per-site raw UTR matrix, so the W index is defined
identically to the mouse notebook.


In [ ]:
adata = ad.read_h5ad(UTR_H5)
obs = adata.obs
uninduced = obs["induced"].astype(str).eq("False") if CONTROLS_ONLY else np.ones(adata.n_obs, bool)
lin = obs["lineage"].astype(str).values
pt_e = pd.to_numeric(obs["dpt_pseudotime_excitatory"], errors="coerce").values
pt_i = pd.to_numeric(obs["dpt_pseudotime_inhibitory"], errors="coerce").values

exc_mask = (
    uninduced.values
    & (lin == "Excitatory neurons")
    & np.isfinite(pt_e)
    & np.isin(obs["celltype"].astype(str), EXC_ORDER)
)
inh_mask = (
    uninduced.values
    & (lin == "Inhibitory neurons")
    & np.isfinite(pt_i)
    & np.isin(obs["celltype"].astype(str), INH_ORDER)
)

X = adata.X.tocsr() if issparse(adata.X) else adata.X
Xe, Xi = X[exc_mask], X[inh_mask]
ct_e = obs["celltype"].astype(str).values[exc_mask]
ct_i = obs["celltype"].astype(str).values[inh_mask]
pte, pti = pt_e[exc_mask], pt_i[inh_mask]
print(f"EXC cells: {Xe.shape[0]} | INH cells: {Xi.shape[0]}")

# per-site gene identity + rank-normalised UTR length (shared across lineages)
site_gene = adata.var["name"].astype(str).values
site_sym = adata.var["gene_name"].astype(str).values
site_size = adata.var["utr_size"].astype(float).values
genes = list(dict.fromkeys(site_gene))
_tmp = defaultdict(list)
for i, g in enumerate(site_gene):
    _tmp[g].append(i)
g2idx = {g: np.array(v) for g, v in _tmp.items()}
gene_sym = {
    g: (g if str(site_sym[idx[0]]) in ("None", "nan", "") else site_sym[idx[0]])
    for g, idx in g2idx.items()
}  # Ensembl fallback for unnamed genes
gene_ranks = {}
for g, idx in g2idx.items():
    K = len(idx)
    r = rankdata(site_size[idx], method="average")
    gene_ranks[g] = (r - 1) / (K - 1) if K > 1 else np.zeros(K)
print(f"multi-site genes: {len(genes)}")
del adata, X

## 3. Core functions (pseudobulk W, correlations)

`W` per (gene, state) from aggregated within-gene PAS proportions (NaN where the gene has < min UMIs in
that state). `pairwise_corr` = symmetric state x state; `cross_corr` = asymmetric (EXC states x INH
states); both ignore NaNs pairwise so small cell types / bins can still contribute.


In [ ]:
def w_profiles(X, labels, states, min_reads):
    """Per-gene pseudobulk W for each state -> (n_genes, n_states), NaN if under-covered."""
    sums = np.vstack(
        [np.asarray(X[labels == s].sum(axis=0)).ravel() for s in states]
    )  # states x sites
    W = np.full((len(genes), len(states)), np.nan)
    for gi, g in enumerate(genes):
        idx = g2idx[g]
        sub = sums[:, idx]
        gsum = sub.sum(axis=1)
        ok = gsum >= min_reads
        if ok.any():
            W[gi, ok] = (sub[ok] / gsum[ok, None]) @ gene_ranks[g]
    return W


def _corr(a, b, method):
    if method == "spearman":
        a, b = rankdata(a), rankdata(b)
    return np.corrcoef(a, b)[0, 1]


def pairwise_corr(P, method="pearson"):
    n = P.shape[1]
    C = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            m = np.isfinite(P[:, i]) & np.isfinite(P[:, j])
            C[i, j] = C[j, i] = (
                _corr(P[m, i], P[m, j], method)
                if (m.sum() >= 3 and P[m, i].std() and P[m, j].std())
                else np.nan
            )
    return C


def cross_corr(A, B, method="pearson"):
    C = np.full((A.shape[1], B.shape[1]), np.nan)
    for i in range(A.shape[1]):
        for j in range(B.shape[1]):
            m = np.isfinite(A[:, i]) & np.isfinite(B[:, j])
            if m.sum() >= 3 and A[m, i].std() and B[m, j].std():
                C[i, j] = _corr(A[m, i], B[m, j], method)
    return C


def color_ticks(ax, labels, axis="both"):
    if axis in ("x", "both"):
        for k, t in enumerate(ax.get_xticklabels()):
            t.set_color(PAL.get(labels[k], "#333333"))
    if axis in ("y", "both"):
        for k, t in enumerate(ax.get_yticklabels()):
            t.set_color(PAL.get(labels[k], "#333333"))


print("functions ready")

## 4. Cross-lineage cell-type map: which stages share a 3'UTRome, and which diverge

W profiles per cell type in each lineage, then the EXC cell types (rows) x INH cell types (columns)
correlation of those profiles (over genes covered in both, pairwise). High = the two lineages have the
same per-gene length pattern at those stages; low = divergence. Cell types are in differentiation
order; expect the progenitor corner (RG-like x vRG-like) to be most similar.


In [ ]:
We_ct = w_profiles(Xe, ct_e, EXC_ORDER, MIN_READS_CT)
Wi_ct = w_profiles(Xi, ct_i, INH_ORDER, MIN_READS_CT)
print(
    "EXC cell types covered per gene (median):",
    int(np.nanmedian(np.isfinite(We_ct).sum(1))),
)

Cx_ct = cross_corr(We_ct, Wi_ct, "pearson")
Cx_ct_sp = cross_corr(We_ct, Wi_ct, "spearman")

### Panel: EXC x INH cell-type UTRome correlation (Pearson base, Spearman variant)

How to read. Rows = excitatory cell types (progenitor oRG at top -> neuron L23 at bottom), columns =
inhibitory cell types (vRG -> CGE_IN). Each cell is the correlation of the two cell types' per-gene
3'UTR-length (W) profiles. Bright = shared length pattern; dark = the lineages length their 3'UTRs
differently at that stage pair. Tick labels are coloured with the organoid palette.


In [ ]:
def crossmap_celltypes(C, corr_name, save_name):
    fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(66) / 300))
    vmn = np.nanmin(C)
    cm = plt.get_cmap(CORR_CMAP).copy()
    cm.set_bad("#dddddd")
    im = ax.imshow(C, cmap=cm, vmin=vmn, vmax=1.0, aspect="auto")
    ax.set_xticks(range(len(INH_ORDER)))
    ax.set_yticks(range(len(EXC_ORDER)))
    ax.set_xticklabels(INH_ORDER, rotation=90, fontsize=S.FONT_TICK)
    ax.set_yticklabels(EXC_ORDER, fontsize=S.FONT_TICK)
    color_ticks(ax, INH_ORDER, "x")
    color_ticks(ax, EXC_ORDER, "y")
    ax.set_xlabel("inhibitory lineage", fontsize=S.FONT_LABEL, color=LIN_COL["INH"])
    ax.set_ylabel("excitatory lineage", fontsize=S.FONT_LABEL, color=LIN_COL["EXC"])
    for i in range(len(EXC_ORDER)):
        for j in range(len(INH_ORDER)):
            if np.isfinite(C[i, j]):
                ax.text(
                    j,
                    i,
                    f"{C[i, j]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=4.0,
                    color="white" if C[i, j] < (vmn + 1) / 2 else "black",
                )
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label(f"{corr_name} r (W profiles)", fontsize=S.FONT_TICK)
    cb.ax.tick_params(labelsize=S.FONT_TICK)
    ax.set_title(f"EXC x INH cell-type 3'UTRome similarity ({corr_name})", fontsize=S.FONT_LABEL)
    fig.tight_layout()
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


crossmap_celltypes(Cx_ct, "Pearson", "exc_inh_utrome_corrmat_celltypes")
crossmap_celltypes(Cx_ct_sp, "Spearman", "exc_inh_utrome_corrmat_celltypes__spearman")

## 5. Cross-lineage pseudotime-bin map

Each lineage is split into `N_BINS` equal-frequency (quantile) pseudotime bins; matched bin index =
matched developmental fraction (bin 0 = progenitors, last bin = mature neurons). The EXC bins (rows) x
INH bins (columns) W-correlation localises divergence to a developmental fraction; axis colour strips
mark each bin's majority cell type (organoid palette). Matching by quantile assumes comparable
developmental progression - a caveat; the cell-type map (Section 4) is the biologically anchored view.


In [ ]:
def quantile_bins(pt, n):
    edges = np.quantile(pt, np.linspace(0, 1, n + 1))
    edges[0] -= 1e-9
    return np.clip(np.digitize(pt, edges) - 1, 0, n - 1)


be, bi = quantile_bins(pte, N_BINS), quantile_bins(pti, N_BINS)
print("EXC bin sizes:", np.bincount(be, minlength=N_BINS))
print("INH bin sizes:", np.bincount(bi, minlength=N_BINS))

We_bin = w_profiles(Xe, be, np.arange(N_BINS), MIN_READS_BIN)
Wi_bin = w_profiles(Xi, bi, np.arange(N_BINS), MIN_READS_BIN)


def bin_majority_rgb(ct, bins):
    maj = [pd.Series(ct[bins == b]).value_counts().idxmax() for b in range(N_BINS)]
    return maj, np.array([to_rgb(PAL.get(c, "#999999")) for c in maj])


maj_e, rgb_e = bin_majority_rgb(ct_e, be)
maj_i, rgb_i = bin_majority_rgb(ct_i, bi)
Cx_bin = cross_corr(We_bin, Wi_bin, "pearson")
Cx_bin_sp = cross_corr(We_bin, Wi_bin, "spearman")
print("EXC bin majority:", maj_e)
print("INH bin majority:", maj_i)


def bin_composition(ct_arr, bins, order, res=140):
    """RGB strip image (res, N_BINS, 3): each bin subdivided top->bottom by its cell-type
    proportions in differentiation order, so every cell type is represented (not just the majority).
    """
    cols = []
    for b in range(N_BINS):
        sub = ct_arr[bins == b]
        vc = pd.Series(sub).value_counts()
        segs = [(cc, vc.get(cc, 0)) for cc in order if vc.get(cc, 0) > 0]
        total = sum(n for _, n in segs) or 1
        col = np.ones((res, 3))
        y0 = 0
        for k, (cc, n) in enumerate(segs):
            h = res - y0 if k == len(segs) - 1 else int(round(n / total * res))
            col[y0 : y0 + h] = to_rgb(PAL.get(cc, "#999999"))
            y0 += h
        cols.append(col)
    return np.stack(cols, axis=1)


comp_e = bin_composition(ct_e, be, EXC_ORDER)
comp_i = bin_composition(ct_i, bi, INH_ORDER)

### Panel: EXC x INH pseudotime-bin UTRome correlation (+ Spearman variant)


In [ ]:
def crossmap_bins(C, corr_name, save_name):
    fig = plt.figure(figsize=(S.HALF_W_PX / 300, S.mm_to_px(78) / 300))
    outer = fig.add_gridspec(
        2,
        2,
        width_ratios=[0.05, 1],
        height_ratios=[0.05, 1],
        wspace=0.03,
        hspace=0.03,
        left=0.20,
        right=0.82,
        top=0.86,
        bottom=0.14,
    )
    fig.add_subplot(outer[0, 0]).axis("off")
    ax_cx = fig.add_subplot(outer[0, 1])  # top strip = INH bins (columns)
    ax_cy = fig.add_subplot(outer[1, 0])  # left strip = EXC bins (rows)
    ax = fig.add_subplot(outer[1, 1])
    vmn = np.nanmin(C)
    cm = plt.get_cmap(CORR_CMAP).copy()
    cm.set_bad("#dddddd")
    im = ax.imshow(C, cmap=cm, vmin=vmn, vmax=1.0, aspect="auto", origin="upper")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("INH pseudotime bin →", fontsize=S.FONT_TICK, color=LIN_COL["INH"])
    ax_cx.imshow(comp_i, aspect="auto")
    ax_cy.imshow(np.transpose(comp_e, (1, 0, 2)), aspect="auto")
    ax_cy.set_ylabel("EXC pseudotime bin →", fontsize=S.FONT_TICK, color=LIN_COL["EXC"], labelpad=6)
    for a in (ax_cx, ax_cy):
        a.set_xticks([])
        a.set_yticks([])
        for s in a.spines.values():
            s.set_visible(False)
    cax = fig.add_axes([0.85, 0.14, 0.02, 0.72])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(f"{corr_name} r", fontsize=S.FONT_TICK)
    cax.tick_params(labelsize=S.FONT_TICK)
    seen = [c for c in EXC_ORDER + INH_ORDER if c in set(ct_e) | set(ct_i)]
    handles = [Patch(facecolor=PAL[c], label=c) for c in seen]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=7,
        frameon=False,
        fontsize=4.6,
        bbox_to_anchor=(0.5, -0.02),
        title="cell-type composition per bin",
        title_fontsize=S.FONT_LEGEND,
    )
    fig.suptitle(
        f"EXC x INH pseudotime-bin 3'UTRome similarity ({corr_name})",
        fontsize=S.FONT_LABEL,
        y=0.98,
    )
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


crossmap_bins(Cx_bin, "Pearson", "exc_inh_utrome_corrmat_pseudotime_bins")
crossmap_bins(Cx_bin_sp, "Spearman", "exc_inh_utrome_corrmat_pseudotime_bins__spearman")

### Panel: within-lineage bin correlations, INH vs EXC (3'UTRome trajectory geometry)

A different view, analogous to the mouse transcriptome-vs-UTRome bin panel - but here the two sides are
the two lineages (same modality, the 3'UTRome W): left = correlations between INH pseudotime bins,
right = correlations between EXC pseudotime bins (each a symmetric bin x bin matrix; top-variance
genes to reveal structure; axis colour strips = bin majority cell type). It shows how each lineage's
3'UTRome evolves along its own pseudotime (band-diagonal = smooth progression; off-diagonal blocks =
stage transitions), so the two trajectory geometries can be compared side by side. (The panel above put
EXC bins x INH bins in a single cross-matrix; this one keeps each lineage on its own axes.) Shared
colour scale, so magnitudes are comparable between the two lineages.


In [ ]:
def _heat_strips(fig, spec, C, comp_img, cmap, vmin, vmax, title, xlabel, xcolor):
    """Symmetric bin x bin heatmap with a cell-type composition strip on both axes
    (each bin subdivided by its cell-type proportions, so every cell type is represented).
    """
    inner = gridspec.GridSpecFromSubplotSpec(
        2,
        2,
        subplot_spec=spec,
        width_ratios=[0.06, 1],
        height_ratios=[0.06, 1],
        wspace=0.03,
        hspace=0.03,
    )
    fig.add_subplot(inner[0, 0]).axis("off")
    ax_cx = fig.add_subplot(inner[0, 1])
    ax_cy = fig.add_subplot(inner[1, 0])
    ax_h = fig.add_subplot(inner[1, 1])
    cm = plt.get_cmap(cmap).copy()
    cm.set_bad("#dddddd")
    im = ax_h.imshow(C, cmap=cm, vmin=vmin, vmax=vmax, aspect="auto", origin="upper")
    ax_h.set_xticks([])
    ax_h.set_yticks([])
    ax_h.set_xlabel(xlabel, fontsize=S.FONT_TICK, color=xcolor)
    ax_cx.imshow(comp_img, aspect="auto")
    ax_cy.imshow(np.transpose(comp_img, (1, 0, 2)), aspect="auto")
    for a in (ax_cx, ax_cy):
        a.set_xticks([])
        a.set_yticks([])
        for sp in a.spines.values():
            sp.set_visible(False)
    ax_cx.set_title(title, fontsize=S.FONT_LABEL, pad=2)
    return im


def within_bins_figure(corr_name, save_name):
    method = corr_name.lower()
    Pi, Pe = Wi_bin[np.isfinite(Wi_bin).all(1)], We_bin[np.isfinite(We_bin).all(1)]
    si = np.argsort(np.nanvar(Pi, axis=1))[::-1][:TOP_N_FEATURES]
    se = np.argsort(np.nanvar(Pe, axis=1))[::-1][:TOP_N_FEATURES]
    Ci, Ce = pairwise_corr(Pi[si], method), pairwise_corr(Pe[se], method)
    vmn = np.nanmin([np.nanmin(Ci), np.nanmin(Ce)])
    fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(94) / 300))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.045], wspace=0.16, top=0.84, bottom=0.20)
    _heat_strips(
        fig,
        gs[0],
        Ci,
        comp_i,
        CORR_CMAP,
        vmn,
        1.0,
        "Inhibitory",
        "INH pseudotime bin →",
        LIN_COL["INH"],
    )
    im = _heat_strips(
        fig,
        gs[1],
        Ce,
        comp_e,
        CORR_CMAP,
        vmn,
        1.0,
        "Excitatory",
        "EXC pseudotime bin →",
        LIN_COL["EXC"],
    )
    cax = fig.add_subplot(gs[2])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(f"{corr_name} r (W profiles)", fontsize=S.FONT_TICK)
    cax.tick_params(labelsize=S.FONT_TICK)
    seen = [c for c in EXC_ORDER + INH_ORDER if c in set(ct_e) | set(ct_i)]
    handles = [Patch(facecolor=PAL[c], label=c) for c in seen]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=7,
        frameon=False,
        fontsize=4.6,
        bbox_to_anchor=(0.5, 0.0),
        title="axis strip = cell-type composition per bin",
        title_fontsize=S.FONT_LEGEND,
    )
    fig.suptitle(
        f"Within-lineage 3'UTRome trajectory geometry: INH vs EXC ({corr_name})",
        fontsize=S.FONT_LABEL,
        y=0.97,
    )
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


within_bins_figure("Pearson", "exc_inh_within_lineage_pseudotime_bins")
within_bins_figure("Spearman", "exc_inh_within_lineage_pseudotime_bins__spearman")

### Panel: within-lineage bins, independent colour scales (diagnostic)

Same INH | EXC within-lineage bin matrices as above, but each lineage is scaled to its own
[min, max] with its own colourbar, so the more compressed lineage's internal bin-to-bin structure is
not washed out by the other's wider range. Trade-off: absolute r values are not comparable across
the two panels here - use the shared-scale version above when comparing magnitudes.


In [ ]:
def _heat_strips_cbar(fig, spec, C, comp_img, cmap, vmin, vmax, title, xlabel, xcolor, cbar_label):
    """Heatmap + composition strips + a colourbar bound tightly to this heatmap (own scale)."""
    inner = gridspec.GridSpecFromSubplotSpec(
        2,
        3,
        subplot_spec=spec,
        width_ratios=[0.06, 1, 0.05],
        height_ratios=[0.06, 1],
        wspace=0.08,
        hspace=0.03,
    )
    fig.add_subplot(inner[0, 0]).axis("off")
    fig.add_subplot(inner[0, 2]).axis("off")
    ax_cx = fig.add_subplot(inner[0, 1])
    ax_cy = fig.add_subplot(inner[1, 0])
    ax_h = fig.add_subplot(inner[1, 1])
    cax = fig.add_subplot(inner[1, 2])
    cm = plt.get_cmap(cmap).copy()
    cm.set_bad("#dddddd")
    im = ax_h.imshow(C, cmap=cm, vmin=vmin, vmax=vmax, aspect="auto", origin="upper")
    ax_h.set_xticks([])
    ax_h.set_yticks([])
    ax_h.set_xlabel(xlabel, fontsize=S.FONT_TICK, color=xcolor)
    ax_cx.imshow(comp_img, aspect="auto")
    ax_cy.imshow(np.transpose(comp_img, (1, 0, 2)), aspect="auto")
    for a in (ax_cx, ax_cy):
        a.set_xticks([])
        a.set_yticks([])
        for sp in a.spines.values():
            sp.set_visible(False)
    ax_cx.set_title(title, fontsize=S.FONT_LABEL, pad=2)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(cbar_label, fontsize=S.FONT_TICK)
    cax.tick_params(labelsize=S.FONT_TICK)
    return im


def within_bins_figure_free(corr_name, save_name):
    """Within-lineage bin correlations, INH | EXC, each on its own colour scale + own colourbar."""
    method = corr_name.lower()
    Pi, Pe = Wi_bin[np.isfinite(Wi_bin).all(1)], We_bin[np.isfinite(We_bin).all(1)]
    si = np.argsort(np.nanvar(Pi, axis=1))[::-1][:TOP_N_FEATURES]
    se = np.argsort(np.nanvar(Pe, axis=1))[::-1][:TOP_N_FEATURES]
    Ci, Ce = pairwise_corr(Pi[si], method), pairwise_corr(Pe[se], method)
    fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(94) / 300))
    outer = gridspec.GridSpec(1, 2, wspace=0.42, top=0.84, bottom=0.20)
    _heat_strips_cbar(
        fig,
        outer[0],
        Ci,
        comp_i,
        CORR_CMAP,
        np.nanmin(Ci),
        np.nanmax(Ci),
        "Inhibitory",
        "INH pseudotime bin →",
        LIN_COL["INH"],
        f"{corr_name} r (INH, own scale)",
    )
    _heat_strips_cbar(
        fig,
        outer[1],
        Ce,
        comp_e,
        CORR_CMAP,
        np.nanmin(Ce),
        np.nanmax(Ce),
        "Excitatory",
        "EXC pseudotime bin →",
        LIN_COL["EXC"],
        f"{corr_name} r (EXC, own scale)",
    )
    seen = [c for c in EXC_ORDER + INH_ORDER if c in set(ct_e) | set(ct_i)]
    handles = [Patch(facecolor=PAL[c], label=c) for c in seen]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=7,
        frameon=False,
        fontsize=4.6,
        bbox_to_anchor=(0.5, 0.0),
        title="axis strip = cell-type composition per bin",
        title_fontsize=S.FONT_LEGEND,
    )
    fig.suptitle(
        f"Within-lineage 3'UTRome geometry: INH vs EXC ({corr_name}, INDEPENDENT scales)",
        fontsize=S.FONT_LABEL,
        y=0.97,
    )
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


within_bins_figure_free("Pearson", "exc_inh_within_lineage_pseudotime_bins__freescale")
within_bins_figure_free("Spearman", "exc_inh_within_lineage_pseudotime_bins__spearman_freescale")

### Panel: within-lineage bins transcriptome, independent scales (direct comparison with UTRome above)

Same plot as the UTRome free-scale panel above, but the modality is the transcriptome
(gene-level expression) instead of the 3'UTR-length W. "Transcriptome" here = log-CPM of each gene's
total 3'-end UMIs (summed over its PAS sites) - the same molecules, cells, genes and bins as the W
analysis, so the only difference is abundance vs within-gene 3'UTR length (not a gene set / cell
difference). Independent per-lineage colour scales. Comparison with the UTRome panel above shows
whether the two lineages' 3'UTRomes change differently from their transcriptomes along pseudotime.


In [ ]:
# Transcriptome = gene-level expression (log-CPM of gene-total UMIs, summed over each gene\'s PAS
# sites) from the same 3\' molecules/cells/genes/bins as the UTRome W - so the only difference vs the
# panel above is abundance (expression) vs within-gene 3\'UTR length.
def expr_profiles(X, labels, states, min_reads):
    sums = np.vstack([np.asarray(X[labels == s].sum(axis=0)).ravel() for s in states])
    lib = sums.sum(axis=1)
    E = np.full((len(genes), len(states)), np.nan)
    for gi, g in enumerate(genes):
        gsum = sums[:, g2idx[g]].sum(axis=1)
        ok = gsum >= min_reads
        E[gi, ok] = np.log1p(gsum[ok] / lib[ok] * 1e6)
    return E


Ee_bin = expr_profiles(Xe, be, np.arange(N_BINS), MIN_READS_BIN)
Ei_bin = expr_profiles(Xi, bi, np.arange(N_BINS), MIN_READS_BIN)


def within_bins_expr_free(corr_name, save_name):
    method = corr_name.lower()
    Pi, Pe = Ei_bin[np.isfinite(Ei_bin).all(1)], Ee_bin[np.isfinite(Ee_bin).all(1)]
    si = np.argsort(np.nanvar(Pi, axis=1))[::-1][:TOP_N_FEATURES]
    se = np.argsort(np.nanvar(Pe, axis=1))[::-1][:TOP_N_FEATURES]
    Ci, Ce = pairwise_corr(Pi[si], method), pairwise_corr(Pe[se], method)
    fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(94) / 300))
    outer = gridspec.GridSpec(1, 2, wspace=0.42, top=0.84, bottom=0.20)
    _heat_strips_cbar(
        fig,
        outer[0],
        Ci,
        comp_i,
        CORR_CMAP,
        np.nanmin(Ci),
        np.nanmax(Ci),
        "Inhibitory",
        "INH pseudotime bin →",
        LIN_COL["INH"],
        f"{corr_name} r (INH, own scale)",
    )
    _heat_strips_cbar(
        fig,
        outer[1],
        Ce,
        comp_e,
        CORR_CMAP,
        np.nanmin(Ce),
        np.nanmax(Ce),
        "Excitatory",
        "EXC pseudotime bin →",
        LIN_COL["EXC"],
        f"{corr_name} r (EXC, own scale)",
    )
    seen = [c for c in EXC_ORDER + INH_ORDER if c in set(ct_e) | set(ct_i)]
    handles = [Patch(facecolor=PAL[c], label=c) for c in seen]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=7,
        frameon=False,
        fontsize=4.6,
        bbox_to_anchor=(0.5, 0.0),
        title="axis strip = cell-type composition per bin",
        title_fontsize=S.FONT_LEGEND,
    )
    fig.suptitle(
        f"Within-lineage TRANSCRIPTOME geometry: INH vs EXC ({corr_name}, INDEPENDENT scales)",
        fontsize=S.FONT_LABEL,
        y=0.97,
    )
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


within_bins_expr_free("Pearson", "exc_inh_within_lineage_pseudotime_bins__transcriptome_freescale")
within_bins_expr_free(
    "Spearman",
    "exc_inh_within_lineage_pseudotime_bins__transcriptome_spearman_freescale",
)

### Variant: quarter-height x full-width, square heatmaps (freescale)

Quarter-height full-width alternatives of the freescale within-lineage corrmats
(3'UTRome-Spearman and transcriptome-Spearman), matching the Fig1 corrmat quarter
variant: both heatmaps forced square, the suptitle removed (its description
belongs in the figure legend), and the 14-cell-type strip key as a compact
vertical legend on the right. The per-heatmap colourbars are kept tight to their
own heatmap (already well placed). Saved as
`..._spearman_freescale__quarter` and `..._transcriptome_spearman_freescale__quarter`.


In [ ]:
# --- Quarter-height full-width square variants of the freescale within-lineage heatmaps ---
# Same treatment as the Fig1 corrmat quarter variant: full width x quarter height
# (S.FULL_W_IN x S.QUARTER_H_IN = 174 x 58.75 mm), both heatmaps forced square, no suptitle
# (description -> fig legend), and the 14-cell-type key as a compact vertical legend on the
# right. Per-heatmap colourbars kept tight to their own heatmap (already well placed).
# Reuses Wi_bin/We_bin, Ei_bin/Ee_bin, comp_i/comp_e, pairwise_corr, PAL, orders, etc.
from matplotlib.patches import Patch

_FIGW_MM, _FIGH_MM = S.FULL_W_MM, S.QUARTER_H_MM


def _square_heat_block_comp(
    fig,
    xleft,
    C,
    comp_img,
    vmin,
    vmax,
    title,
    xlabel,
    xcolor,
    cbar_label,
    *,
    side=45.0,
    ybot=7.0,
    strip=1.8,
    sgap=0.3,
    tstrip=2.0,
    tgap=0.6,
    cgap=1.5,
    cw=1.8,
    cbar_frac=1.0,
):
    """Square (side x side mm) bin-corr heatmap + cell-type composition strips + its own
    colourbar, placed via explicit mm->figure-fraction axes so the heatmap is exactly square
    and the strips/colourbar stay tight and aligned. `xleft` = left edge of the y-strip (mm).
    """
    fx = lambda v: v / _FIGW_MM
    fy = lambda v: v / _FIGH_MM
    hx = xleft + strip + sgap
    ax_h = fig.add_axes([fx(hx), fy(ybot), fx(side), fy(side)])
    cm = plt.get_cmap(CORR_CMAP).copy()
    cm.set_bad("#dddddd")
    im = ax_h.imshow(C, cmap=cm, vmin=vmin, vmax=vmax, aspect="auto", origin="upper")
    ax_h.set_xticks([])
    ax_h.set_yticks([])
    ax_h.set_xlabel(xlabel, fontsize=S.FONT_TICK, color=xcolor, labelpad=2)
    ax_cy = fig.add_axes([fx(xleft), fy(ybot), fx(strip), fy(side)])
    ax_cy.imshow(np.transpose(comp_img, (1, 0, 2)), aspect="auto")
    ax_cx = fig.add_axes([fx(hx), fy(ybot + side + tgap), fx(side), fy(tstrip)])
    ax_cx.imshow(comp_img, aspect="auto")
    for a in (ax_cy, ax_cx):
        a.set_xticks([])
        a.set_yticks([])
        for s in a.spines.values():
            s.set_visible(False)
    ax_cx.set_title(title, fontsize=S.FONT_LABEL, pad=2)
    cax = fig.add_axes([fx(hx + side + cgap), fy(ybot), fx(cw), fy(side * cbar_frac)])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(cbar_label, fontsize=S.FONT_TICK, labelpad=1)
    cax.tick_params(labelsize=S.FONT_TICK, pad=1)
    return im


def within_bins_free_quarter(Pi_bin, Pe_bin, corr_name, save_name, cbar_frac=1.0):
    method = corr_name.lower()
    Pi = Pi_bin[np.isfinite(Pi_bin).all(1)]
    Pe = Pe_bin[np.isfinite(Pe_bin).all(1)]
    si = np.argsort(np.nanvar(Pi, axis=1))[::-1][:TOP_N_FEATURES]
    se = np.argsort(np.nanvar(Pe, axis=1))[::-1][:TOP_N_FEATURES]
    Ci, Ce = pairwise_corr(Pi[si], method), pairwise_corr(Pe[se], method)
    fig = plt.figure(figsize=(S.FULL_W_IN, S.QUARTER_H_IN))
    _square_heat_block_comp(
        fig,
        4.0,
        Ci,
        comp_i,
        np.nanmin(Ci),
        np.nanmax(Ci),
        "Inhibitory",
        "INH pseudotime bin →",
        LIN_COL["INH"],
        f"{corr_name} r (INH)",
        cbar_frac=cbar_frac,
    )
    _square_heat_block_comp(
        fig,
        75.0,
        Ce,
        comp_e,
        np.nanmin(Ce),
        np.nanmax(Ce),
        "Excitatory",
        "EXC pseudotime bin →",
        LIN_COL["EXC"],
        f"{corr_name} r (EXC)",
        cbar_frac=cbar_frac,
    )
    seen = [c for c in EXC_ORDER + INH_ORDER if c in set(ct_e) | set(ct_i)]
    handles = [Patch(facecolor=PAL[c], label=c) for c in seen]
    fig.legend(
        handles=handles,
        loc="center left",
        bbox_to_anchor=(0.80, 0.5),
        ncol=1,
        frameon=False,
        fontsize=S.FONT_LEGEND - 0.5,
        handletextpad=0.4,
        labelspacing=0.22,
        title="axis strip:\ncell type",
        title_fontsize=S.FONT_LEGEND_TITLE - 0.5,
    )
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


within_bins_free_quarter(
    Wi_bin,
    We_bin,
    "Spearman",
    "exc_inh_within_lineage_pseudotime_bins__spearman_freescale__quarter",
)
within_bins_free_quarter(
    Ei_bin,
    Ee_bin,
    "Spearman",
    "exc_inh_within_lineage_pseudotime_bins__transcriptome_spearman_freescale__quarter",
)

In [ ]:
# --- Short-colourbar variants: same quarter within-lineage heatmaps, correlation colourbar
# much shorter (cbar_frac=0.4) + bottom-aligned -> lower-right, outside the heatmaps. ---
within_bins_free_quarter(
    Wi_bin,
    We_bin,
    "Spearman",
    "exc_inh_within_lineage_pseudotime_bins__spearman_freescale__quarter_shortcbar",
    cbar_frac=0.4,
)
within_bins_free_quarter(
    Ei_bin,
    Ee_bin,
    "Spearman",
    "exc_inh_within_lineage_pseudotime_bins__transcriptome_spearman_freescale__quarter_shortcbar",
    cbar_frac=0.4,
)

In [ ]:
# --- Eighth-height x half-width square variants of the freescale within-lineage heatmaps ---
# Both heatmaps kept square in HALF_W x EIGHTH_H (85 x 27.375 mm). Blocks pushed apart + short
# 1-char colourbar label so the INH colourbar/ticks clear the EXC heatmap; 2-col tiny legend.
_OFW, _OFH = _FIGW_MM, _FIGH_MM
_FIGW_MM, _FIGH_MM = S.HALF_W_MM, S.EIGHTH_H_MM


def within_bins_free_eighth_half(Pi_bin, Pe_bin, corr_name, save_name):
    method = corr_name.lower()
    Pi = Pi_bin[np.isfinite(Pi_bin).all(1)]
    Pe = Pe_bin[np.isfinite(Pe_bin).all(1)]
    si = np.argsort(np.nanvar(Pi, axis=1))[::-1][:TOP_N_FEATURES]
    se = np.argsort(np.nanvar(Pe, axis=1))[::-1][:TOP_N_FEATURES]
    Ci, Ce = pairwise_corr(Pi[si], method), pairwise_corr(Pe[se], method)
    fig = plt.figure(figsize=(S.HALF_W_IN, S.EIGHTH_H_IN))
    kw = dict(
        side=14.0,
        ybot=4.8,
        strip=1.3,
        sgap=0.2,
        tstrip=1.2,
        tgap=0.45,
        cgap=1.0,
        cw=1.2,
        cbar_frac=0.6,
    )
    _square_heat_block_comp(
        fig,
        2.5,
        Ci,
        comp_i,
        np.nanmin(Ci),
        np.nanmax(Ci),
        "Inhibitory",
        "INH pt bin →",
        LIN_COL["INH"],
        "r",
        **kw,
    )
    _square_heat_block_comp(
        fig,
        28.0,
        Ce,
        comp_e,
        np.nanmin(Ce),
        np.nanmax(Ce),
        "Excitatory",
        "EXC pt bin →",
        LIN_COL["EXC"],
        "r",
        **kw,
    )
    seen = [c for c in EXC_ORDER + INH_ORDER if c in set(ct_e) | set(ct_i)]
    handles = [Patch(facecolor=PAL[c], label=c) for c in seen]
    fig.legend(
        handles=handles,
        loc="center left",
        bbox_to_anchor=(0.62, 0.5),
        ncol=2,
        frameon=False,
        fontsize=S.FONT_LEGEND - 1.5,
        handletextpad=0.3,
        columnspacing=0.5,
        labelspacing=0.15,
        handlelength=0.9,
        handleheight=0.9,
    )
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


within_bins_free_eighth_half(
    Wi_bin,
    We_bin,
    "Spearman",
    "exc_inh_within_lineage_pseudotime_bins__spearman_freescale__eighth_half",
)
within_bins_free_eighth_half(
    Ei_bin,
    Ee_bin,
    "Spearman",
    "exc_inh_within_lineage_pseudotime_bins__transcriptome_spearman_freescale__eighth_half",
)
_FIGW_MM, _FIGH_MM = _OFW, _OFH

In [ ]:
# --- Quarter-height x half-width variant with square heatmaps made as large as possible by
# moving the cell-type legend to a horizontal strip under each heatmap (INH types under the INH
# heatmap, EXC types under the EXC heatmap) -> frees the right side that the vertical legend used. ---
from matplotlib.patches import Patch as _Patch

_OFW2, _OFH2 = _FIGW_MM, _FIGH_MM
_FIGW_MM, _FIGH_MM = S.HALF_W_MM, S.QUARTER_H_MM


def within_bins_free_quarter_half_hlegend(Pi_bin, Pe_bin, corr_name, save_name):
    method = corr_name.lower()
    Pi = Pi_bin[np.isfinite(Pi_bin).all(1)]
    Pe = Pe_bin[np.isfinite(Pe_bin).all(1)]
    si = np.argsort(np.nanvar(Pi, axis=1))[::-1][:TOP_N_FEATURES]
    se = np.argsort(np.nanvar(Pe, axis=1))[::-1][:TOP_N_FEATURES]
    Ci, Ce = pairwise_corr(Pi[si], method), pairwise_corr(Pe[se], method)
    fig = plt.figure(figsize=(S.HALF_W_IN, S.QUARTER_H_IN))
    SIDE = 30.0
    XL_I, XL_E = 2.5, 43.0
    kw = dict(
        side=SIDE,
        ybot=17.0,
        strip=1.4,
        sgap=0.25,
        tstrip=1.5,
        tgap=0.5,
        cgap=0.9,
        cw=1.2,
        cbar_frac=0.45,
    )
    _square_heat_block_comp(
        fig,
        XL_I,
        Ci,
        comp_i,
        np.nanmin(Ci),
        np.nanmax(Ci),
        "Inhibitory",
        "INH pt bin →",
        LIN_COL["INH"],
        f"{corr_name} r",
        **kw,
    )
    _square_heat_block_comp(
        fig,
        XL_E,
        Ce,
        comp_e,
        np.nanmin(Ce),
        np.nanmax(Ce),
        "Excitatory",
        "EXC pt bin →",
        LIN_COL["EXC"],
        f"{corr_name} r",
        **kw,
    )
    # horizontal cell-type legends under each heatmap (split by lineage)
    hx_i = (XL_I + 1.4 + 0.25 + SIDE / 2) / S.HALF_W_MM
    hx_e = (XL_E + 1.4 + 0.25 + SIDE / 2) / S.HALF_W_MM
    inh_ct = [c for c in INH_ORDER if c in set(ct_i)]
    exc_ct = [c for c in EXC_ORDER if c in set(ct_e)]
    li = fig.legend(
        [_Patch(facecolor=PAL[c]) for c in inh_ct],
        inh_ct,
        loc="upper center",
        bbox_to_anchor=(hx_i, 0.155),
        ncol=3,
        frameon=False,
        fontsize=S.FONT_LEGEND - 2,
        handletextpad=0.25,
        columnspacing=0.5,
        labelspacing=0.15,
        handlelength=0.8,
        handleheight=0.8,
    )
    fig.add_artist(li)
    fig.legend(
        [_Patch(facecolor=PAL[c]) for c in exc_ct],
        exc_ct,
        loc="upper center",
        bbox_to_anchor=(hx_e, 0.155),
        ncol=4,
        frameon=False,
        fontsize=S.FONT_LEGEND - 2,
        handletextpad=0.25,
        columnspacing=0.5,
        labelspacing=0.15,
        handlelength=0.8,
        handleheight=0.8,
    )
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


within_bins_free_quarter_half_hlegend(
    Wi_bin,
    We_bin,
    "Spearman",
    "exc_inh_within_lineage_pseudotime_bins__spearman_freescale__quarter_half_hlegend",
)
within_bins_free_quarter_half_hlegend(
    Ei_bin,
    Ee_bin,
    "Spearman",
    "exc_inh_within_lineage_pseudotime_bins__transcriptome_spearman_freescale__quarter_half_hlegend",
)
_FIGW_MM, _FIGH_MM = _OFW2, _OFH2

### Variant: inhibitory-only, half-width x quarter-height (UTRome Spearman freescale)

The inhibitory heatmap alone for an inhibitory-only Fig 2: single square heatmap at half
width x quarter height (85 x 58.75 mm), composition strips + its own colourbar tight to the
heatmap, and a compact INH cell-type legend on the right. (No transcriptome version.)
Saved as `exc_inh_within_lineage_pseudotime_bins__spearman_freescale__inh_quarter`.


In [ ]:
# --- Inhibitory-only half-width x quarter-height variant (UTRome Spearman freescale) ---
# Single INH heatmap for an inhibitory-only Fig 2: square, half width x quarter height, with
# composition strips + own colourbar tight to the heatmap and a compact INH cell-type legend on
# the right. Reuses Wi_bin, comp_i, pairwise_corr, INH_ORDER, PAL, LIN_COL, CORR_CMAP, etc.
from matplotlib.patches import Patch


def _square_block_hw(
    fig,
    xleft,
    C,
    comp_img,
    vmin,
    vmax,
    title,
    xlabel,
    xcolor,
    cbar_label,
    figW,
    figH,
    *,
    side=40.0,
    ybot=7.0,
    strip=1.8,
    sgap=0.3,
    tstrip=2.0,
    tgap=0.6,
    cgap=1.5,
    cw=1.8,
):
    """Square heatmap + composition strips + own colourbar (explicit mm axes). Returns the
    colourbar's right edge (mm) so the legend can be placed clear of it."""
    fx = lambda v: v / figW
    fy = lambda v: v / figH
    hx = xleft + strip + sgap
    ax_h = fig.add_axes([fx(hx), fy(ybot), fx(side), fy(side)])
    cm = plt.get_cmap(CORR_CMAP).copy()
    cm.set_bad("#dddddd")
    im = ax_h.imshow(C, cmap=cm, vmin=vmin, vmax=vmax, aspect="auto", origin="upper")
    ax_h.set_xticks([])
    ax_h.set_yticks([])
    ax_h.set_xlabel(xlabel, fontsize=S.FONT_TICK, color=xcolor, labelpad=2)
    ax_cy = fig.add_axes([fx(xleft), fy(ybot), fx(strip), fy(side)])
    ax_cy.imshow(np.transpose(comp_img, (1, 0, 2)), aspect="auto")
    ax_cx = fig.add_axes([fx(hx), fy(ybot + side + tgap), fx(side), fy(tstrip)])
    ax_cx.imshow(comp_img, aspect="auto")
    for a in (ax_cy, ax_cx):
        a.set_xticks([])
        a.set_yticks([])
        for s in a.spines.values():
            s.set_visible(False)
    ax_cx.set_title(title, fontsize=S.FONT_LABEL, pad=2)
    cax = fig.add_axes([fx(hx + side + cgap), fy(ybot), fx(cw), fy(side)])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(cbar_label, fontsize=S.FONT_TICK, labelpad=1)
    cax.tick_params(labelsize=S.FONT_TICK, pad=1)
    return hx + side + cgap + cw


_Pi = Wi_bin[np.isfinite(Wi_bin).all(1)]
_si = np.argsort(np.nanvar(_Pi, axis=1))[::-1][:TOP_N_FEATURES]
_Ci = pairwise_corr(_Pi[_si], "spearman")
_figW, _figH = S.HALF_W_MM, S.QUARTER_H_MM
fig = plt.figure(figsize=(S.HALF_W_IN, S.QUARTER_H_IN))
_right = _square_block_hw(
    fig,
    4.0,
    _Ci,
    comp_i,
    np.nanmin(_Ci),
    np.nanmax(_Ci),
    "Inhibitory",
    "INH pseudotime bin →",
    LIN_COL["INH"],
    "Spearman r",
    _figW,
    _figH,
    side=40.0,
)
_handles = [Patch(facecolor=PAL[c], label=c) for c in INH_ORDER]
fig.legend(
    handles=_handles,
    loc="center left",
    bbox_to_anchor=((_right + 9) / _figW, 0.5),
    ncol=1,
    frameon=False,
    fontsize=S.FONT_LEGEND - 0.5,
    handletextpad=0.4,
    labelspacing=0.3,
    title="axis strip:\ncell type",
    title_fontsize=S.FONT_LEGEND_TITLE - 0.5,
)
S.save_and_show(
    fig,
    "exc_inh_within_lineage_pseudotime_bins__spearman_freescale__inh_quarter",
    fig_dir=UNASSIGNED_DIR,
)

### Variant: excitatory-only, half-width x quarter-height (UTRome Spearman freescale)

Excitatory counterpart of the inhibitory-only heatmap above: single square heatmap at half
width x quarter height, composition strips + own tight colourbar, compact EXC cell-type legend
on the right. Saved as `exc_inh_within_lineage_pseudotime_bins__spearman_freescale__exc_quarter`.


In [ ]:
# --- Excitatory-only half-width x quarter-height variant (UTRome Spearman freescale) ---
# EXC counterpart of the inhibitory-only heatmap. Reuses _square_block_hw + Patch defined above,
# and We_bin, comp_e, pairwise_corr, EXC_ORDER, PAL, LIN_COL, TOP_N_FEATURES.
_Pe = We_bin[np.isfinite(We_bin).all(1)]
_se = np.argsort(np.nanvar(_Pe, axis=1))[::-1][:TOP_N_FEATURES]
_Ce = pairwise_corr(_Pe[_se], "spearman")
_figW, _figH = S.HALF_W_MM, S.QUARTER_H_MM
fig = plt.figure(figsize=(S.HALF_W_IN, S.QUARTER_H_IN))
_right = _square_block_hw(
    fig,
    4.0,
    _Ce,
    comp_e,
    np.nanmin(_Ce),
    np.nanmax(_Ce),
    "Excitatory",
    "EXC pseudotime bin →",
    LIN_COL["EXC"],
    "Spearman r",
    _figW,
    _figH,
    side=40.0,
)
_handles = [Patch(facecolor=PAL[c], label=c) for c in EXC_ORDER]
fig.legend(
    handles=_handles,
    loc="center left",
    bbox_to_anchor=((_right + 9) / _figW, 0.5),
    ncol=1,
    frameon=False,
    fontsize=S.FONT_LEGEND - 0.5,
    handletextpad=0.4,
    labelspacing=0.28,
    title="axis strip:\ncell type",
    title_fontsize=S.FONT_LEGEND_TITLE - 0.5,
)
S.save_and_show(
    fig,
    "exc_inh_within_lineage_pseudotime_bins__spearman_freescale__exc_quarter",
    fig_dir=UNASSIGNED_DIR,
)

### Panel: rate of change of transcriptome vs 3'UTRome (Mantel + decay + within-stage coherence)

Quantifies the visual bin comparison above.

Top row - decay curves: for each modality, the mean
bin-to-bin correlation as a function of the pseudotime separation (lag) between bins. A curve that stays
high = the modality changes slowly/smoothly along the trajectory; a curve that drops fast = it changes
sharply. Overlaying transcriptome (green) and 3'UTRome (purple) per lineage shows whether they change at
different rates. The Mantel test in each title asks whether the two modalities' whole bin-to-bin
distance geometries agree: r near 1 = same trajectory shape, lower r = decoupled (permutation p from
shuffling bins).

Bottom row - within-stage similarity: each bin's correlation with its immediate
neighbours, plotted along pseudotime; this localises where the two modalities decouple. Main read-out
for the mature-neuron end (right side of each plot): where the transcriptome stays high (adjacent
late/neuron bins look alike) but the 3'UTRome drops (adjacent late bins differ), the 3'UTRome is
carrying cell-state variation the transcriptome does not - the decoupling flagged for mature INH neurons.


In [ ]:
TX_COL, UTR_COL = "#3B7A57", "#7B3294"  # transcriptome (green) vs 3'UTRome (purple)


def _topvar(P, n=TOP_N_FEATURES):
    Pc = P[np.isfinite(P).all(1)]
    return Pc[np.argsort(np.nanvar(Pc, axis=1))[::-1][:n]]


def _mantel(D1, D2, n_perm=9999, seed=0):
    iu = np.triu_indices_from(D1, 1)
    a, b = D1[iu], D2[iu]
    m = np.isfinite(a) & np.isfinite(b)
    a, b = a[m], b[m]
    r = np.corrcoef(a, b)[0, 1]
    rng = np.random.default_rng(seed)
    n = D1.shape[0]
    cnt = 0
    for _ in range(n_perm):
        p = rng.permutation(n)
        cnt += abs(np.corrcoef(a, D2[np.ix_(p, p)][iu][m])[0, 1]) >= abs(r)
    return r, (cnt + 1) / (n_perm + 1)


def _decay(C):
    n = C.shape[0]
    return np.arange(1, n), np.array(
        [np.nanmean([C[i, i + d] for i in range(n - d)]) for d in range(1, n)]
    )


def _localcoh(C):
    n = C.shape[0]
    return np.array([np.nanmean([C[i, j] for j in (i - 1, i + 1) if 0 <= j < n]) for i in range(n)])


Ct_i, Cu_i = (
    pairwise_corr(_topvar(Ei_bin)),
    pairwise_corr(_topvar(Wi_bin)),
)  # INH: transcriptome, UTRome
Ct_e, Cu_e = pairwise_corr(_topvar(Ee_bin)), pairwise_corr(_topvar(We_bin))  # EXC

fig, axes = plt.subplots(2, 2, figsize=(S.FULL_W_PX / 300, S.mm_to_px(122) / 300))
mres = {}
for col, (Ct, Cu, name, lcol) in enumerate(
    [
        (Ct_i, Cu_i, "Inhibitory", LIN_COL["INH"]),
        (Ct_e, Cu_e, "Excitatory", LIN_COL["EXC"]),
    ]
):
    r, p = _mantel(1 - Ct, 1 - Cu)
    mres[name] = (r, p)
    lags, dt = _decay(Ct)
    _, du = _decay(Cu)
    ax = axes[0, col]
    ax.plot(lags, dt, "-o", ms=3, lw=1.3, color=TX_COL, label="Transcriptome")
    ax.plot(lags, du, "-o", ms=3, lw=1.3, color=UTR_COL, label="3'UTRome (W)")
    ax.set_title(
        f"{name}: change vs bin separation\nMantel(Tx, UTRome) r={r:.2f}, p={p:.3f}",
        fontsize=S.FONT_LABEL,
        color=lcol,
    )
    ax.set_xlabel("pseudotime-bin separation (lag)", fontsize=S.FONT_LABEL)
    ax.set_ylabel("mean correlation", fontsize=S.FONT_LABEL)
    ax.legend(frameon=False, fontsize=S.FONT_LEGEND)
    ax.tick_params(labelsize=S.FONT_TICK)
    ax2 = axes[1, col]
    ax2.plot(
        range(len(Ct)),
        _localcoh(Ct),
        "-o",
        ms=3,
        lw=1.3,
        color=TX_COL,
        label="Transcriptome",
    )
    ax2.plot(
        range(len(Cu)),
        _localcoh(Cu),
        "-o",
        ms=3,
        lw=1.3,
        color=UTR_COL,
        label="3'UTRome (W)",
    )
    ax2.set_title(
        f"{name}: within-stage similarity along pseudotime",
        fontsize=S.FONT_LABEL,
        color=lcol,
    )
    ax2.set_xlabel("pseudotime bin (progenitor → neuron)", fontsize=S.FONT_LABEL)
    ax2.set_ylabel("corr with neighbouring bins", fontsize=S.FONT_LABEL)
    ax2.legend(frameon=False, fontsize=S.FONT_LEGEND)
    ax2.tick_params(labelsize=S.FONT_TICK)
fig.suptitle(
    "Transcriptome vs 3'UTRome: how differently they change along each lineage",
    fontsize=S.FONT_LABEL,
    y=1.0,
)
fig.tight_layout()
S.save_and_show(fig, "exc_inh_transcriptome_vs_utrome_decay_mantel", fig_dir=UNASSIGNED_DIR)
print(
    "Mantel (transcriptome vs UTRome bin geometry):",
    {k: f"r={v[0]:.2f}, p={v[1]:.3f}" for k, v in mres.items()},
)

### Panel: within-stage similarity with bootstrap 95% CI (bins) + overall coupling

Cell-level bootstrap (B=250) of the decoupling: within each pseudotime bin we resample
its cells with replacement and recompute the per-bin transcriptome and 3'UTRome profiles (feature set
fixed to the point estimate, so only cell-sampling noise varies), giving a sampling distribution for the
within-stage similarity at each bin and for the overall Mantel coupling. First figure: within-stage
similarity (transcriptome green vs 3'UTRome purple) with 95% CI, for INH and EXC. Second figure:
the overall transcriptome-vs-3'UTRome bin-geometry coupling (Mantel r) per lineage with 95% CI - lower =
more decoupled; INH sits well below EXC and below 1.

Bands are centred on the point estimate and span +/-1.96 x the bootstrap standard error. We do not use
the raw 2.5-97.5 percentile band: resampling cells with replacement (~63% unique) adds noise that
slightly decorrelates neighbouring bins, so the bootstrap statistic is biased a little low and the raw
percentile band is shifted below the full-data curve. Using the bootstrap for the spread (SE) only and
centring on the full-data point estimate is the standard bootstrap-SE interval.


In [ ]:
import scipy.sparse as sp

# vectorised pseudobulk over all genes (verified == w_profiles) so many bootstraps are cheap
_gi = {g: i for i, g in enumerate(genes)}
_site_gidx = np.array([_gi[g] for g in site_gene])
_Smap = sp.csr_matrix(
    (np.ones(len(_site_gidx)), (np.arange(len(_site_gidx)), _site_gidx)),
    shape=(len(_site_gidx), len(genes)),
)
_rankvec = np.zeros(len(site_gene))
for _g, _idx in g2idx.items():
    _rankvec[_idx] = gene_ranks[_g]


def fast_profiles(site_sums, min_reads):
    gene_tot = np.asarray(_Smap.T @ site_sums.T)
    w_num = np.asarray(_Smap.T @ (site_sums * _rankvec[None, :]).T)
    lib = site_sums.sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        W = w_num / gene_tot
        E = np.log1p(gene_tot / lib[None, :] * 1e6)
    under = gene_tot < min_reads
    W[under] = np.nan
    E[under] = np.nan
    return W, E


def topvar_idx(P, n=TOP_N_FEATURES):
    cov = np.isfinite(P).all(1)
    ic = np.where(cov)[0]
    return ic[np.argsort(np.nanvar(P[cov], axis=1))[::-1][:n]]


def _state_sums(X, labels, states, sel=False, rng=None):
    rows = []
    for s in states:
        cs = np.where(labels == s)[0]
        if sel:
            cs = rng.choice(cs, size=len(cs), replace=True)
        rows.append(np.asarray(X[cs].sum(axis=0)).ravel())
    return np.vstack(rows)


def boot_stats(X, labels, states, min_reads, B=250, seed=0, want_mantel=False):
    """Within-stage similarity (Tx & UTRome) point + bootstrap SE; optional Mantel(Tx,UTRome) point + SE.
    Bootstrap = resample cells with replacement within each state (feature set fixed to point estimate).
    """
    W0, E0 = fast_profiles(_state_sums(X, labels, states), min_reads)
    uix, tix = topvar_idx(W0), topvar_idx(E0)
    Cu0, Ct0 = pairwise_corr(W0[uix]), pairwise_corr(E0[tix])
    iu = np.triu_indices(len(states), 1)
    r0 = np.corrcoef((1 - Ct0)[iu], (1 - Cu0)[iu])[0, 1] if want_mantel else np.nan
    rng = np.random.default_rng(seed)
    ut_b, tx_b, r_b = [], [], []
    for _ in range(B):
        Wb, Eb = fast_profiles(_state_sums(X, labels, states, sel=True, rng=rng), min_reads)
        Cub, Ctb = pairwise_corr(Wb[uix]), pairwise_corr(Eb[tix])
        ut_b.append(_localcoh(Cub))
        tx_b.append(_localcoh(Ctb))
        if want_mantel:
            r_b.append(np.corrcoef((1 - Ctb)[iu], (1 - Cub)[iu])[0, 1])
    d = dict(
        tx=_localcoh(Ct0),
        ut=_localcoh(Cu0),
        tx_sd=np.nanstd(tx_b, 0),
        ut_sd=np.nanstd(ut_b, 0),
    )
    if want_mantel:
        d.update(r=r0, r_sd=np.nanstd(r_b))
    return d


bs_i = boot_stats(Xi, bi, np.arange(N_BINS), MIN_READS_BIN, want_mantel=True)
bs_e = boot_stats(Xe, be, np.arange(N_BINS), MIN_READS_BIN, want_mantel=True)
print(
    f"Mantel(Tx,UTRome) bins +/-95%CI: INH {bs_i['r']:.2f} +/- {1.96 * bs_i['r_sd']:.2f} | "
    f"EXC {bs_e['r']:.2f} +/- {1.96 * bs_e['r_sd']:.2f}"
)


def _plot_within_ci(pairs, xlabels, xlabel, suptitle, save_name, rot=0, colorx=False):
    fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, S.mm_to_px(68) / 300))
    for ax, (bs, name, lcol, order) in zip(axes, pairs):
        x = np.arange(len(bs["tx"]))
        for pt, sd, col, lab in [
            (bs["tx"], bs["tx_sd"], TX_COL, "Transcriptome"),
            (bs["ut"], bs["ut_sd"], UTR_COL, "3'UTRome (W)"),
        ]:
            ax.fill_between(x, pt - 1.96 * sd, pt + 1.96 * sd, color=col, alpha=0.25, lw=0)
            ax.plot(x, pt, "-o", ms=3, lw=1.3, color=col, label=lab)
        if order is None:
            ax.set_xlabel(xlabel, fontsize=S.FONT_LABEL)
        else:
            ax.set_xticks(x)
            ax.set_xticklabels(order, rotation=rot, fontsize=S.FONT_TICK)
            if colorx:
                color_ticks(ax, order, "x")
        ax.set_ylabel(xlabels, fontsize=S.FONT_LABEL)
        ax.set_title(f"{name} (95% bootstrap CI)", fontsize=S.FONT_LABEL, color=lcol)
        ax.legend(frameon=False, fontsize=S.FONT_LEGEND)
        ax.tick_params(labelsize=S.FONT_TICK)
    fig.suptitle(suptitle, fontsize=S.FONT_LABEL, y=1.0)
    fig.tight_layout()
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


_plot_within_ci(
    [
        (bs_i, "Inhibitory", LIN_COL["INH"], None),
        (bs_e, "Excitatory", LIN_COL["EXC"], None),
    ],
    "corr with neighbouring bins",
    "pseudotime bin (progenitor → neuron)",
    "Within-stage similarity along pseudotime: transcriptome vs 3'UTRome (95% CI)",
    "exc_inh_within_stage_bins_bootstrap_ci",
)

# overall coupling (Mantel r) per lineage with 95% CI
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(62) / 300))
xs, rs, es, cols, names = (
    [0, 1],
    [bs_i["r"], bs_e["r"]],
    [1.96 * bs_i["r_sd"], 1.96 * bs_e["r_sd"]],
    [LIN_COL["INH"], LIN_COL["EXC"]],
    ["Inhibitory", "Excitatory"],
)
ax.errorbar(xs, rs, yerr=es, fmt="none", ecolor="k", capsize=5, lw=1.3)
for xi, ri, ci in zip(xs, rs, cols):
    ax.plot(xi, ri, "o", ms=8, color=ci)
    ax.annotate(
        f"{ri:.2f}",
        (xi, ri),
        textcoords="offset points",
        xytext=(9, 0),
        fontsize=S.FONT_TICK,
        va="center",
    )
ax.axhline(1.0, color="gray", ls=":", lw=0.7)
ax.set_xticks(xs)
ax.set_xticklabels(names, fontsize=S.FONT_TICK)
for t, ci in zip(ax.get_xticklabels(), cols):
    t.set_color(ci)
ax.set_ylim(0, 1.05)
ax.set_xlim(-0.5, 1.5)
ax.set_ylabel("Mantel r (Tx vs UTRome geometry)", fontsize=S.FONT_LABEL)
ax.set_title(
    "Overall coupling per lineage\n(lower = more decoupled; 1 = identical; 95% CI)",
    fontsize=S.FONT_LABEL,
)
ax.tick_params(labelsize=S.FONT_TICK)
fig.tight_layout()
S.save_and_show(fig, "exc_inh_decoupling_mantel_bootstrap", fig_dir=UNASSIGNED_DIR)

### Panel: within-lineage cell-type correlations, INH vs EXC (all cell types as states)

Same idea as the within-lineage bin panel above, but the states are cell types (pseudobulk W per
cell type) rather than pseudotime bins, so every cell type of each lineage is an explicit
row/column and nothing is hidden behind a bin's majority label. Left = INH cell types x INH cell types,
right = EXC cell types x EXC cell types (differentiation order; bright = shared per-gene 3'UTR-length
pattern; values annotated). This is the within-lineage counterpart of the EXC x INH cross-map (Section
4). Cell types overlap heavily in pseudotime, which is why several never became a bin-majority above.


In [ ]:
def within_ct_figure(corr_name, save_name):
    method = corr_name.lower()
    Ci, Ce = pairwise_corr(Wi_ct, method), pairwise_corr(We_ct, method)  # within-lineage cell types
    vmn = np.nanmin([np.nanmin(Ci), np.nanmin(Ce)])
    cm = plt.get_cmap(CORR_CMAP).copy()
    cm.set_bad("#dddddd")
    fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(80) / 300))
    gs = gridspec.GridSpec(
        1,
        3,
        width_ratios=[len(INH_ORDER), len(EXC_ORDER), 0.5],
        wspace=0.45,
        top=0.86,
        bottom=0.22,
    )
    im = None
    for gi, (C, order, ttl, lcol) in enumerate(
        [
            (Ci, INH_ORDER, "Inhibitory", LIN_COL["INH"]),
            (Ce, EXC_ORDER, "Excitatory", LIN_COL["EXC"]),
        ]
    ):
        ax = fig.add_subplot(gs[gi])
        im = ax.imshow(C, cmap=cm, vmin=vmn, vmax=1.0, aspect="auto")
        ax.set_xticks(range(len(order)))
        ax.set_yticks(range(len(order)))
        ax.set_xticklabels(order, rotation=90, fontsize=S.FONT_TICK)
        ax.set_yticklabels(order, fontsize=S.FONT_TICK)
        color_ticks(ax, order, "both")
        ax.set_title(ttl, fontsize=S.FONT_LABEL, color=lcol)
        mid = (vmn + 1) / 2
        for ii in range(len(order)):
            for jj in range(len(order)):
                if np.isfinite(C[ii, jj]):
                    ax.text(
                        jj,
                        ii,
                        f"{C[ii, jj]:.2f}",
                        ha="center",
                        va="center",
                        fontsize=3.8,
                        color="white" if C[ii, jj] < mid else "black",
                    )
    cax = fig.add_subplot(gs[2])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(f"{corr_name} r (W profiles)", fontsize=S.FONT_TICK)
    cax.tick_params(labelsize=S.FONT_TICK)
    fig.suptitle(
        f"Within-lineage 3'UTRome by cell type: INH vs EXC ({corr_name})",
        fontsize=S.FONT_LABEL,
        y=0.98,
    )
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


within_ct_figure("Pearson", "exc_inh_within_lineage_celltypes")
within_ct_figure("Spearman", "exc_inh_within_lineage_celltypes__spearman")

### Panel: within-lineage cell-type correlations - transcriptome (compare with UTRome cell-type panel above)

The transcriptome counterpart of the cell-type UTRome panel above (states = cell types; matched
gene-total expression, same genes/cells as the W analysis). Left = INH cell types x INH, right = EXC x
EXC. Read against the UTRome cell-type panel, it shows whether cell types that look alike
transcriptionally also look alike in 3'UTR length. If e.g. the mature inhibitory interneuron types are
highly correlated here (similar transcriptome) but less correlated in the UTRome panel, their 3'UTRomes
are decoupled from their transcriptomes.


In [ ]:
Ei_ct = expr_profiles(Xi, ct_i, INH_ORDER, MIN_READS_CT)
Ee_ct = expr_profiles(Xe, ct_e, EXC_ORDER, MIN_READS_CT)


def within_ct_corr_figure(Pi_ct, Pe_ct, corr_name, save_name, modality, cbar_label):
    method = corr_name.lower()
    Ci, Ce = pairwise_corr(Pi_ct, method), pairwise_corr(Pe_ct, method)
    vmn = np.nanmin([np.nanmin(Ci), np.nanmin(Ce)])
    cm = plt.get_cmap(CORR_CMAP).copy()
    cm.set_bad("#dddddd")
    fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(80) / 300))
    gs = gridspec.GridSpec(
        1,
        3,
        width_ratios=[len(INH_ORDER), len(EXC_ORDER), 0.5],
        wspace=0.45,
        top=0.86,
        bottom=0.22,
    )
    im = None
    for gi, (C, order, ttl, lcol) in enumerate(
        [
            (Ci, INH_ORDER, "Inhibitory", LIN_COL["INH"]),
            (Ce, EXC_ORDER, "Excitatory", LIN_COL["EXC"]),
        ]
    ):
        ax = fig.add_subplot(gs[gi])
        im = ax.imshow(C, cmap=cm, vmin=vmn, vmax=1.0, aspect="auto")
        ax.set_xticks(range(len(order)))
        ax.set_yticks(range(len(order)))
        ax.set_xticklabels(order, rotation=90, fontsize=S.FONT_TICK)
        ax.set_yticklabels(order, fontsize=S.FONT_TICK)
        color_ticks(ax, order, "both")
        ax.set_title(ttl, fontsize=S.FONT_LABEL, color=lcol)
        mid = (vmn + 1) / 2
        for ii in range(len(order)):
            for jj in range(len(order)):
                if np.isfinite(C[ii, jj]):
                    ax.text(
                        jj,
                        ii,
                        f"{C[ii, jj]:.2f}",
                        ha="center",
                        va="center",
                        fontsize=3.8,
                        color="white" if C[ii, jj] < mid else "black",
                    )
    cax = fig.add_subplot(gs[2])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(cbar_label, fontsize=S.FONT_TICK)
    cax.tick_params(labelsize=S.FONT_TICK)
    fig.suptitle(
        f"Within-lineage {modality} by cell type: INH vs EXC ({corr_name})",
        fontsize=S.FONT_LABEL,
        y=0.98,
    )
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


within_ct_corr_figure(
    Ei_ct,
    Ee_ct,
    "Pearson",
    "exc_inh_within_lineage_celltypes__transcriptome",
    "TRANSCRIPTOME",
    "Pearson r (expression)",
)
within_ct_corr_figure(
    Ei_ct,
    Ee_ct,
    "Spearman",
    "exc_inh_within_lineage_celltypes__transcriptome_spearman",
    "TRANSCRIPTOME",
    "Spearman r (expression)",
)

### Panel: within-stage similarity by cell type, with bootstrap 95% CI (transcriptome vs 3'UTRome; no Mantel)

The cell-type analogue of the bins CI panel: for each cell type (differentiation order) its mean
correlation with its neighbouring cell types, transcriptome (green) vs 3'UTRome (purple), per
lineage, with 95% bootstrap CI (resample cells within each cell type; band = point estimate +/-1.96 x
bootstrap SE, as above). Where the transcriptome band stays high but the 3'UTRome band dips, adjacent
cell types look alike transcriptionally yet differ in 3'UTR length. Mantel omitted here.
Cell-type pseudobulk averages over many cells, so the decoupling is subtler than at the bin level.


In [ ]:
bs_ct_i = boot_stats(Xi, ct_i, INH_ORDER, MIN_READS_CT)
bs_ct_e = boot_stats(Xe, ct_e, EXC_ORDER, MIN_READS_CT)
_plot_within_ci(
    [
        (bs_ct_i, "Inhibitory", LIN_COL["INH"], INH_ORDER),
        (bs_ct_e, "Excitatory", LIN_COL["EXC"], EXC_ORDER),
    ],
    "corr with neighbouring cell types",
    None,
    "Within-stage similarity by cell type: transcriptome vs 3'UTRome (95% CI)",
    "exc_inh_within_stage_similarity_celltypes",
    rot=90,
    colorx=True,
)

### Panel: within-stage similarity, INH vs EXC on a shared y-axis (bins + cell type)

The same within-stage CI curves as above, but with INH and EXC forced onto one shared y-scale (EXC
y-tick labels removed, since identical to INH) so the magnitude of the INH 3'UTRome drop can be read
directly against EXC. Reuses the bootstrap results already computed (no re-sampling). Two figures:
pseudotime bins and cell types.


In [ ]:
def _plot_within_ci_sharedy(pairs, ylabel_text, xlabel, suptitle, save_name, rot=0, colorx=False):
    fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, S.mm_to_px(68) / 300), sharey=True)
    for k, (ax, (bs, name, lcol, order)) in enumerate(zip(axes, pairs)):
        x = np.arange(len(bs["tx"]))
        for pt, sd, col, lab in [
            (bs["tx"], bs["tx_sd"], TX_COL, "Transcriptome"),
            (bs["ut"], bs["ut_sd"], UTR_COL, "3'UTRome (W)"),
        ]:
            ax.fill_between(x, pt - 1.96 * sd, pt + 1.96 * sd, color=col, alpha=0.25, lw=0)
            ax.plot(x, pt, "-o", ms=3, lw=1.3, color=col, label=lab)
        if order is None:
            ax.set_xlabel(xlabel, fontsize=S.FONT_LABEL)
        else:
            ax.set_xticks(x)
            ax.set_xticklabels(order, rotation=rot, fontsize=S.FONT_TICK)
            if colorx:
                color_ticks(ax, order, "x")
        ax.set_title(f"{name} (95% bootstrap CI)", fontsize=S.FONT_LABEL, color=lcol)
        ax.legend(frameon=False, fontsize=S.FONT_LEGEND)
        ax.tick_params(labelsize=S.FONT_TICK)
        if k == 0:
            ax.set_ylabel(ylabel_text, fontsize=S.FONT_LABEL)
        else:
            ax.tick_params(labelleft=False)  # EXC shares INH's y-axis
    fig.suptitle(suptitle, fontsize=S.FONT_LABEL, y=1.0)
    fig.tight_layout()
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


_plot_within_ci_sharedy(
    [
        (bs_i, "Inhibitory", LIN_COL["INH"], None),
        (bs_e, "Excitatory", LIN_COL["EXC"], None),
    ],
    "corr with neighbouring bins",
    "pseudotime bin (progenitor → neuron)",
    "Within-stage similarity (bins), shared y-axis: transcriptome vs 3'UTRome (95% CI)",
    "exc_inh_within_stage_bins_bootstrap_ci__sharedy",
)

_plot_within_ci_sharedy(
    [
        (bs_ct_i, "Inhibitory", LIN_COL["INH"], INH_ORDER),
        (bs_ct_e, "Excitatory", LIN_COL["EXC"], EXC_ORDER),
    ],
    "corr with neighbouring cell types",
    None,
    "Within-stage similarity by cell type, shared y-axis: transcriptome vs 3'UTRome (95% CI)",
    "exc_inh_within_stage_similarity_celltypes__sharedy",
    rot=90,
    colorx=True,
)

## 6. Global 3'UTR-length trajectory per lineage

On the shared, well-covered gene set (covered in every matched bin of both lineages), the global
3'UTR-length index = the mean per-gene W in each bin. Plotting EXC and INH on the matched developmental
axis shows where the overall length diverges and which lineage is longer. (Same gene set for both
lineages, so differences are length differences, not gene-set artefacts.)


In [ ]:
common = np.isfinite(We_bin).all(1) & np.isfinite(Wi_bin).all(1)
print(f"genes covered in ALL {N_BINS} bins of BOTH lineages: {common.sum()}")
gW_e = np.nanmean(We_bin[common], axis=0)
gW_i = np.nanmean(Wi_bin[common], axis=0)

fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(60) / 300))
x = np.arange(N_BINS)
ax.plot(x, gW_e, "-o", color=LIN_COL["EXC"], ms=3, lw=1.4, label="Excitatory")
ax.plot(x, gW_i, "-o", color=LIN_COL["INH"], ms=3, lw=1.4, label="Inhibitory")
ax.set_xlabel("matched pseudotime bin (progenitor → neuron)", fontsize=S.FONT_LABEL)
ax.set_ylabel("global 3'UTR-length index (mean W)", fontsize=S.FONT_LABEL)
ax.set_title(
    f"Global 3'UTR length per lineage ({common.sum()} shared genes)",
    fontsize=S.FONT_LABEL,
)
ax.legend(frameon=False, fontsize=S.FONT_LEGEND)
ax.tick_params(labelsize=S.FONT_TICK)
fig.tight_layout()
S.save_and_show(fig, "exc_inh_global_utr_length_trajectory", fig_dir=UNASSIGNED_DIR)

## 7. Divergence by stage: similarity and magnitude along matched pseudotime

Two complementary curves on the shared gene set: the matched-bin cross-lineage correlation of W
profiles (pattern similarity; the diagonal of Section 5) and the mean |Delta-W| per matched bin
(magnitude of per-gene length difference). Where similarity drops and |Delta-W| rises = the stage the
two lineages' 3'UTRomes diverge.


In [ ]:
diag_corr = np.array(
    [cross_corr(We_bin[common][:, [b]], Wi_bin[common][:, [b]])[0, 0] for b in range(N_BINS)]
)
dW = We_bin[common] - Wi_bin[common]  # genes x bins (EXC - INH)
mean_absdW = np.nanmean(np.abs(dW), axis=0)

fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(60) / 300))
x = np.arange(N_BINS)
ax.plot(
    x,
    diag_corr,
    "-o",
    color="#444444",
    ms=3,
    lw=1.3,
    label="W-profile corr (similarity)",
)
ax.set_xlabel("matched pseudotime bin (progenitor → neuron)", fontsize=S.FONT_LABEL)
ax.set_ylabel("cross-lineage W corr", fontsize=S.FONT_LABEL)
ax.tick_params(labelsize=S.FONT_TICK)
ax2 = ax.twinx()
ax2.plot(x, mean_absdW, "-s", color="#B2182B", ms=3, lw=1.3, label="mean |ΔW| (magnitude)")
ax2.set_ylabel("mean |ΔW| per gene", fontsize=S.FONT_LABEL, color="#B2182B")
ax2.tick_params(labelsize=S.FONT_TICK, colors="#B2182B")
lines = ax.get_lines() + ax2.get_lines()
ax.legend(
    lines,
    [ln.get_label() for ln in lines],
    frameon=False,
    fontsize=S.FONT_LEGEND,
    loc="center left",
)
ax.set_title("Where the two 3'UTRomes diverge along development", fontsize=S.FONT_LABEL)
fig.tight_layout()
S.save_and_show(fig, "exc_inh_utrome_divergence_by_stage", fig_dir=UNASSIGNED_DIR)
print(
    f"most divergent matched bin: {int(np.argmax(mean_absdW))} "
    f"(min similarity at bin {int(np.argmin(diag_corr))})"
)

## 8. Per-gene Delta-W: which genes diverge, and where

`Delta-W(gene, bin) = W_EXC - W_INH` on matched bins (shared gene set). Positive (red) = EXC keeps a
longer 3'UTR than INH at that stage; negative (blue) = INH longer. The heatmap shows the top genes by
peak |Delta-W|, revealing both the divergent genes and the developmental stage where they split. Full
table saved for follow-up.


In [ ]:
try:
    from adjustText import adjust_text

    HAS_AT = True
except ImportError:
    HAS_AT = False

cg = np.array(genes)[common]
peak = np.nanmax(np.abs(dW), axis=1)
dwdf = pd.DataFrame(
    {
        "gene_id": cg,
        "gene_name": [gene_sym[g] for g in cg],
        "peak_absdW": peak,
        "peak_bin": np.nanargmax(np.abs(dW), axis=1),
        "mean_dW": np.nanmean(dW, axis=1),
    }
)
dwdf.to_csv(RESULTS_DIR / "exc_inh_deltaW_per_gene.csv", index=False)
top = dwdf.sort_values("peak_absdW", ascending=False).head(30)
print(f"saved {RESULTS_DIR / 'exc_inh_deltaW_per_gene.csv'}")
print("top divergent genes (EXC-longer +, INH-longer -):")
print(top[["gene_name", "peak_bin", "mean_dW", "peak_absdW"]].head(12).to_string(index=False))

M = dW[top.index.values]  # 30 x N_BINS
vmax = np.nanpercentile(np.abs(M), 98)
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(96) / 300))
im = ax.imshow(M, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xlabel("matched pseudotime bin (progenitor → neuron)", fontsize=S.FONT_LABEL)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top["gene_name"], fontsize=4.6, style="italic")
ax.set_xticks(range(N_BINS))
ax.set_xticklabels(range(N_BINS), fontsize=S.FONT_TICK)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
cb.set_label("ΔW = W(EXC) - W(INH)", fontsize=S.FONT_TICK)
cb.ax.tick_params(labelsize=S.FONT_TICK)
ax.set_title("Top divergent genes: EXC vs INH 3'UTR length by stage", fontsize=S.FONT_LABEL)
fig.tight_layout()
S.save_and_show(fig, "exc_inh_per_gene_deltaW_localization", fig_dir=UNASSIGNED_DIR)

## 9. Summary and outputs

Panels -> `Fig2/unassigned_panels/` (Pearson = base, Spearman = `__spearman`); per-gene Delta-W table ->
`utrcalling/lineage_utrome_comparison/`. Final cell regenerates the Figure-2 panel overview HTML.


In [ ]:
print("Panels saved to:", UNASSIGNED_DIR)
for p in sorted(UNASSIGNED_DIR.glob("exc_inh_*.png")):
    print("  -", p.name)
print("\nWhere do the EXC and INH 3'UTRomes diverge?")
print(
    f"  cell-type map: progenitor corner similarity "
    f"(oRG x vRG r={Cx_ct[0, 0]:.2f}); neuron corner (L23 x CGE_IN r={Cx_ct[-1, -1]:.2f})"
)
print(
    f"  matched-bin similarity: {diag_corr[0]:.2f} (bin 0) -> {diag_corr[-1]:.2f} (bin {N_BINS - 1}); "
    f"mean|ΔW| peaks at bin {int(np.argmax(mean_absdW))}"
)
print(
    f"  global length: EXC {gW_e[0]:.3f}->{gW_e[-1]:.3f}, INH {gW_i[0]:.3f}->{gW_i[-1]:.3f} "
    f"over development"
)
print(f"  {len(dwdf)} shared genes scored; top divergent: {', '.join(top['gene_name'].head(6))}")